<a href="https://colab.research.google.com/github/Mridener/IN498-Capstone-in-Analytics/blob/main/IN498_MakaylaRidener_Unit4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GitHub Repository

A GitHub repository was used to track project progress and store all assignment files.

In [15]:
!git add IN498_Unit4.txt

!git commit -m "Added Unit 4 analysis output"

!git push

[main 3a0f86f] Added Unit 4 analysis output
 1 file changed, 59 insertions(+)
 create mode 100644 Unit4/IN498_Unit4.txt
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 790 bytes | 790.00 KiB/s, done.
Total 4 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Mridener/IN498-Capstone-in-Analytics.git
   b77c393..3a0f86f  main -> main


# Decision Tree and Random Forest Code

The final CSV file was read into a data frame and used for the decision tree and random forest data analysis.

In [14]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn import metrics
import sys

#Ignoring warnings
if not sys.warnoptions:
    import warnings
    warnings.simplefilter("ignore")


#Set the writeFunction boolean
#True = write to concole
#False = write to file for assignment turnin
PRINT = False

#Open a file handle for assignment results.
if PRINT == False:
    f = open("IN498_Unit4.txt", "w")
###############################################
##
##PURPOSE: Write to console or file based on
##  the writeFunction variable
##      True = write to concole
##      False = write to file for assignment turnin
##
##INPUT: Message to write (message1)
##   Optional: message2
##
##OUTPUT: None
##
###############################################
def writeFunction(message1, *message2):

    #Print to console
    if PRINT:
        print(message1)
        print(message2)
        print()
    #Print to file for assignment
    else:
        f.write(str(message1))
        f.write(str(message2))
        f.write("\n\n")


#Widen the column display
pd.set_option('max_colwidth',500)

#Read data into a DataFrame using these columns
##"Date","Package_Name","Country","Store_Listing_Visitors",
##"Installers","Visitor-to-Installer_conversion_rate",
##"Installers_retained_for_1_day","Installer-to-1_day_retention_rate",
##"Installers_retained_for_7_days","Installer-to-7_days_retention_rate",
##"Installers_retained_for_15_days","Installer-to-15_days_retention_rate",
##"Installers_retained_for_30_days","Installer-to-30_days_retention_rate"
col_names = ["Date",
             "Package_Name",
             "Country",
             "Store_Listing_Visitors",
             "Installers",
             "Visitor-to-Installer_conversion_rate",
             "Installers_retained_for_1_day",
             "Installer-to-1_day_retention_rate",
             "Installers_retained_for_7_days",
             "Installer-to-7_days_retention_rate",
             "Installers_retained_for_15_days",
             "Installer-to-15_days_retention_rate",
             "Installers_retained_for_30_days",
             "Installer-to-30_days_retention_rate"]

data = pd.read_csv("retained_installers_final.csv")

############ FIX MISSING DATA #######################
#Replace NaN with 0 for Installers_retained_for_30_days
data["Installers_retained_for_30_days"] = data["Installers_retained_for_30_days"].fillna(0)

#################################### INSTALL_30 ##########################################
#Add a new column for installers retained for 30 days
#  If greater than 0, put 1, if 0, put 0
data["Install_30"] = np.where(data["Installers_retained_for_30_days"] > 0, 1, 0)

#Create X using Installers and y using Install_30 columns
X = data[["Installers"]]
y = data["Install_30"]

#Train/test split 80% train, 20% test
#Save into these variables: X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

########### DECISION TREE #######################
#Build a decision tree model
tree = DecisionTreeClassifier(random_state=42)

#Fit the tree using X_train and y_train
tree.fit(X_train, y_train)

#Print the accuracy of the tree using X_train and y_train
writeFunction("Decision Tree Accuracy")
writeFunction(tree.score(X_train, y_train))


########### RANDOM FOREST #######################
#Build a random forest tree model
# n_estimators = 1000
rf = RandomForestClassifier(n_estimators=1000,
                            random_state=42)

#Fit the random tree model with X_train and y_train
rf.fit(X_train, y_train)

#Predict using X_test and the random forest model and store in y_pred
y_pred = rf.predict(X_test)

#Print the random forest results using y_pred
writeFunction("Random Forest Results")
writeFunction(y_pred)

#Print the accuracy for the random forest model using y_test and y_pred
writeFunction("Random Forest Accuracy")
writeFunction(metrics.accuracy_score(y_test, y_pred))

#Predict retention over 30 days for number of installs
#  Use 1,2,4
writeFunction("Random Forest Prediction for 1 Install")
writeFunction(rf.predict(pd.DataFrame({"Installers": [1]})))

writeFunction("Random Forest Prediction for 2 Installs")
writeFunction(rf.predict(pd.DataFrame({"Installers": [2]})))

writeFunction("Random Forest Prediction for 4 Installs")
writeFunction(rf.predict(pd.DataFrame({"Installers": [4]})))

#Get the absolute errors for the random forest model
# Use y_pred and y_test
errors = abs(y_pred - y_test)

#Print the absolute errors for the random forest model
# Use y_pred and y_test
writeFunction("Random Forest Absolute Errors")
writeFunction(errors)

# Print out the mean absolute error
writeFunction("Mean Absolute Error")
writeFunction(np.mean(errors))

#Get the predicitons using X_test and save to rf_probs
rf_probs = rf.predict_proba(X_test)[:, 1]

#Print the predictions for Random Forest using rf_probs
writeFunction("Random Forest Probabilities")
writeFunction(rf_probs)

#Compute Area Under the Receiver Operating Characteristic Curve
#Get the ROC AUC score for the random forest model for y_test and rf_probs
roc_auc = roc_auc_score(y_test, rf_probs)

#Print the ROC AUC for the random forest model
writeFunction("Random Forest ROC AUC")
writeFunction(roc_auc)

#Get the mean absolute percentage error (MAPE) using y_test
mape = np.mean(np.where(y_test != 0, errors / y_test, 0)) * 100

#Print the mean absolute percentage error (MAPE) using y_test
writeFunction("Mean Absolute Percentage Error")
writeFunction(mape)

#Get the accuracy for the random forest model using mape
accuracy = 100 - mape

#Print the accuracy for the random forest model
writeFunction("Random Forest Accuracy Using MAPE")
writeFunction(accuracy)

#Close the file handle
if PRINT == False:
    f.close()




# Decision Tree and Random Forest Results

**Area Under the Receiver Operating Characteristic (ROC) Curve**

The ROC AUC measures how well a classification model separates two groups. A value closer to 1 means the model is better at making correct predictions and a value closer to 0.5 means the predictions are closer to random guessing. The ROC AUC is used to evaluate classification models because it measures how well a model separates the classes instead of only looking at overall accuracy. The random forest model produced a ROC AUC score of 0.9293, showing it was able to separate retained and non-retained users after 30 days with a high level of accuracy.

**Accuracy and Prediction Results**

The decision tree model produced an accuracy of 88.88%. The random forest model produced an accuracy of 88.86% and a ROC AUC score of 0.9293. The random forest model predicted that one install would not result in 30-day retention, and that two and four installs would result in 30-day retention. Even though the decision tree model had a slightly higher accuracy, the random forest model was the better model because it also had a ROC AUC score of 0.9293. Overall, both models showed the same trend. As the number of installs increased, the likelihood of 30-day retention also increased.